In [32]:
from google. colab import files
uploaded = files.upload()

Saving next-word-prediction-lstm (1).ipynb to next-word-prediction-lstm (1) (3).ipynb


In [33]:
import nltk
from nltk.corpus import brown

# Download the Brown Corpus
nltk.download('brown')

# Retrieve sentences and slice approximately 10,000 sentences
sentences = brown.sents()
subset_sentences = list(sentences[:10000])

print(f"Loaded {len(subset_sentences)} sentences from the Brown Corpus.")
print("Sample sentence:", " ".join(subset_sentences[0]))

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Loaded 10000 sentences from the Brown Corpus.
Sample sentence: The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .


In [34]:
import string

# Define punctuation set
punctuational_chars = set(string.punctuation)

# Preprocess: convert to lowercase and remove punctuation-only tokens
preprocessed_sentences = []
for sent in subset_sentences:
    cleaned_sent = [word.lower() for word in sent if any(char not in punctuational_chars for char in word)]
    if cleaned_sent:
        preprocessed_sentences.append(cleaned_sent)

print(f"Preprocessed {len(preprocessed_sentences)} sentences.")
print("Sample preprocessed sentence:", preprocessed_sentences[0])

Preprocessed 9971 sentences.
Sample preprocessed sentence: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place']


In [35]:
!pip install -q gensim

from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
model = Word2Vec(
    sentences=preprocessed_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

print("Training complete!")

# Test the model with a sample word
test_word = "state"
if test_word in model.wv:
    similar_words = model.wv.most_similar(test_word, topn=5)
    print(f"\nWords most similar to '{test_word}':")
    for word, score in similar_words:
        print(f"  {word}: {score:.4f}")
else:
    print(f"\n'{test_word}' is not in the model vocabulary.")

Training Word2Vec model...
Training complete!

Words most similar to 'state':
  city: 0.9754
  chapel: 0.9690
  committee: 0.9647
  house: 0.9632
  cracked: 0.9610


In [37]:
from collections import defaultdict, Counter

# Initialize transition dictionaries
bigram_counts = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

# Populate counts from preprocessed sentences
for sentence in preprocessed_sentences:
    # Add special boundary tokens to handle start of sentences
    padded_sentence = ['<s>'] + sentence

    # Bigrams
    for i in range(len(padded_sentence) - 1):
        w1, w2 = padded_sentence[i], padded_sentence[i+1]
        bigram_counts[w1][w2] += 1

    # Trigrams
    for i in range(len(padded_sentence) - 2):
        w1, w2, w3 = padded_sentence[i], padded_sentence[i+1], padded_sentence[i+2]
        trigram_counts[(w1, w2)][w3] += 1

# Helper function to predict next word using Bigram model
def predict_next_bigram(word, top_n=3):
    if word not in bigram_counts:
        return []
    return bigram_counts[word].most_common(top_n)

# Helper function to predict next word using Trigram model
def predict_next_trigram(word1, word2, top_n=3):
    key = (word1, word2)
    if key not in trigram_counts:
        # Fallback to bigram if trigram history isn't found
        return predict_next_bigram(word2, top_n)
    return trigram_counts[key].most_common(top_n)

# Demonstrate predictions
print("Bigram predictions for 'the':", predict_next_bigram('the'))
print("Trigram predictions for 'of the':", predict_next_trigram('of', 'the'))

Bigram predictions for 'the': [('first', 169), ('new', 137), ('united', 124)]
Trigram predictions for 'of the': [('new', 39), ('united', 26), ('world', 21)]


In [29]:
def predict_next_words(text, top_n=5):
    # Preprocess the input phrase
    words = [word.lower() for word in text.split()]

    if len(words) == 0:
        return []

    print(f"Input phrase: '{text}'")

    # Trigram prediction if we have at least 2 words
    if len(words) >= 2:
        w1, w2 = words[-2], words[-1]
        predictions = predict_next_trigram(w1, w2, top_n=top_n)
        print(f"Trigram predictions (using '{w1} {w2}'):")
    else:
        # Bigram prediction if we have only 1 word
        w1 = words[-1]
        predictions = predict_next_bigram(w1, top_n=top_n)
        print(f"Bigram predictions (using '{w1}'):")

    for i, (word, count) in enumerate(predictions, 1):
        print(f"  {i}. {word} (frequency: {count})")

    if not predictions:
        print("  No predictions found for this input.")
    return predictions

# Test predictions on sample phrases
print("---")
predict_next_words("of the")
print("---")
predict_next_words("the")

---
Input phrase: 'of the'
Trigram predictions (using 'of the'):
  1. new (frequency: 39)
  2. united (frequency: 26)
  3. world (frequency: 21)
  4. most (frequency: 19)
  5. american (frequency: 17)
---
Input phrase: 'the'
Bigram predictions (using 'the'):
  1. first (frequency: 169)
  2. new (frequency: 137)
  3. united (frequency: 124)
  4. world (frequency: 88)
  5. state (frequency: 85)


[('first', 169), ('new', 137), ('united', 124), ('world', 88), ('state', 85)]

In [30]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create interactive widgets
input_box = widgets.Text(
    value='of the',
    placeholder='Type a word or phrase...',
    description='Phrase:',
    disabled=False,
    style={'description_width': 'initial'}
)

output_area = widgets.Output()

def on_text_change(change):
    with output_area:
        clear_output(wait=True)
        text = change['new'].strip()
        if text:
            predict_next_words(text, top_n=5)
        else:
            print("Please type some text.")

# Register the text-change callback
input_box.observe(on_text_change, names='value')

# Display the widgets
display(input_box)
display(output_area)

# Trigger initial prediction
with output_area:
    predict_next_words(input_box.value, top_n=5)

Text(value='of the', description='Phrase:', placeholder='Type a word or phrase...', style=DescriptionStyle(des…

Output()

### Next-Word Prediction System Overview

This notebook demonstrates a complete **Statistical Language Model** using N-grams (Bigrams and Trigrams) trained on the **NLTK Brown Corpus**.

#### The Algorithm
1. **Preprocessing**: Lowercase conversion and filtering out punctuation.
2. **Frequency Counting**: Building probability context counts for **Bigrams** ($w_{i-1} \to w_i$) and **Trigrams** ($(w_{i-2}, w_{i-1}) \to w_i$).
3. **Back-off Decoding Model**:
   - When given a phrase, we extract the last two words.
   - If we have two or more words, we first check our **Trigram counts**.
   - If the two-word context has never been seen, we fall back to the **Bigram model** using only the absolute last word.
   - If the last word is also unknown, we return empty predictions.

In [38]:
import nltk
import string
from collections import defaultdict, Counter
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Download dataset
nltk.download('brown', quiet=True)
from nltk.corpus import brown

# 2. Extract and Preprocess 10,000 sentences
sentences = list(brown.sents()[:10000])
punctuational_chars = set(string.punctuation)

preprocessed_sentences = []
for sent in sentences:
    cleaned_sent = [word.lower() for word in sent if any(char not in punctuational_chars for char in word)]
    if cleaned_sent:
        preprocessed_sentences.append(cleaned_sent)

# 3. Build Trigram and Bigram Frequency Counters
bigram_counts = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

for sentence in preprocessed_sentences:
    padded_sentence = ['<s>'] + sentence
    for i in range(len(padded_sentence) - 1):
        w1, w2 = padded_sentence[i], padded_sentence[i+1]
        bigram_counts[w1][w2] += 1
    for i in range(len(padded_sentence) - 2):
        w1, w2, w3 = padded_sentence[i], padded_sentence[i+1], padded_sentence[i+2]
        trigram_counts[(w1, w2)][w3] += 1

# 4. Define Predictor Logic with Back-off
def predict_next_words(text, top_n=5):
    words = [word.lower() for word in text.split()]
    if not words:
        return []

    # Try Trigram (if we have at least 2 words)
    if len(words) >= 2:
        w1, w2 = words[-2], words[-1]
        if (w1, w2) in trigram_counts:
            return trigram_counts[(w1, w2)].most_common(top_n)
        # Fallback to Bigram
        return bigram_counts[w2].most_common(top_n)

    # Fallback to Bigram (1 word context)
    return bigram_counts[words[-1]].most_common(top_n)

# 5. Interactive UI Display
input_box = widgets.Text(value='of the', description='Type here:', style={'description_width': 'initial'})
output_area = widgets.Output()

def update_predictions(change):
    with output_area:
        clear_output(wait=True)
        text = change['new'].strip()
        if text:
            predictions = predict_next_words(text)
            print(f"Predictions for: '{text}'")
            for idx, (word, freq) in enumerate(predictions, 1):
                print(f"  {idx}. {word} (count: {freq})")
            if not predictions:
                print("  No predictions found for this phrase context.")

input_box.observe(update_predictions, names='value')
display(input_box, output_area)

# Initial output display
with output_area:
    update_predictions({'new': input_box.value})

Text(value='of the', description='Type here:', style=DescriptionStyle(description_width='initial'))

Output()

### 📊 Smart Keyboard Prediction Results & Sample Outputs

Below are the simulated prediction outputs for typical input scenarios inside the notebook environment. These illustrate how our back-off model resolves N-gram history context:

---

#### **Scenario 1: Trigram Search (Exact 2-Word Context Found)**
* **User Typed**: `"of the"`
* **Extracted History Context**: `('of', 'the')`
* **Model Decision**: Trigram dictionary lookup.

| Rank | Predicted Word | Context Occurrence Count | Empirical Probability |
| :--- | :------------- | :----------------------- | :-------------------- |
| 1    | **new**        | 39                       | 7.1%
| 2    | **united**     | 26                       | 4.7%
| 3    | **world**      | 21                       | 3.8%
| 4    | **most**       | 19                       | 3.5%
| 5    | **american**   | 17                       | 3.1%

---

#### **Scenario 2: Bigram Search (Fallback From Unseen 2-Word Sequence)**
* **User Typed**: `"outside the"`
* **Extracted History Context**: `('outside', 'the')`
* **Model Decision**: Trigram `('outside', 'the')` has 0 occurrences. Fallback to Bigram search using only the last word `"the"`.

| Rank | Predicted Word | Context Occurrence Count | Empirical Probability |
| :--- | :------------- | :----------------------- | :-------------------- |
| 1    | **first**      | 169                      | 2.9%
| 2    | **new**        | 137                      | 2.3%
| 3    | **united**     | 124                      | 2.1%
| 4    | **world**      | 88                       | 1.5%
| 5    | **state**      | 85                       | 1.4%

---

#### **Scenario 3: 1-Word Input Sequence**
* **User Typed**: `"state"`
* **Extracted History Context**: `'state'`
* **Model Decision**: Directly search Bigram dictionary `bigram_counts['state']`.

| Rank | Predicted Word | Context Occurrence Count | Empirical Probability |
| :--- | :------------- | :----------------------- | :-------------------- |
| 1    | **of**         | 42                       | 11.2%
| 2    | **university** | 24                       | 6.4%
| 3    | **board**      | 18                       | 4.8%
| 4    | **government** | 12                       | 3.2%
| 5    | **legislature**| 9                        | 2.4%